# `nlp_scoring_exploration.ipynb` — V2
# DSO 1 — NLP Scoring Model · Exploration & Baseline
### Projet ALIA — Avatar Visite Médicale VITAL SA · TDSP Phase 3

**Contexte :**  
Ce notebook explore le dataset DS4 V2 (600 conversations, 16 produits VITAL réels)  
et établit les benchmarks pour les **6 tâches NLP** du système de scoring ALIA.

**Les 6 tâches :**

| Tâche | Type | Target | Source PDFs |
|-------|------|--------|-------------|
| T1 | Classification | Qualité réponse (Excellent/Bon/Faible) | Manuel VITAL |
| T2 | Régression multi-output | Scores scientifique/clarté/objection (0–10) | Manuel VITAL |
| T3 | Classification | Sentiment client (Positif/Neutre/Négatif) | Manuel VITAL |
| T4 | Classification | Niveau ALIA (Débutant/Junior/Confirmé/Expert) | Référentiel 4 niveaux |
| T5 | Classification | Format visite (Flash/Standard/Approfondie) | Manuel — Section 5 |
| T6 | Classification binaire | Conformité (conforme / non-conforme) | Référentiel + Annexe V2 |

**Nouveautés V2 :**
- 14 nouvelles features linguistiques (A-C-R-V, BIP, conformité, 6 étapes)
- 16 produits VITAL réels (Annexe V2)
- Labels niveau ALIA alignés sur le Référentiel officiel
- Détection automatique mots tueurs et surpromesses

**Pipeline :**
```
DS4 V2 (600 conversations)
    └── EDA & distributions
        └── NLPFeatureExtractor V2 (24 features)
            └── Baselines T1→T6 (SVM / Ridge / LogReg)
                └── Benchmark & sélection modèle
                    └── nlp_scoring_train.py V2
```


# 1. Imports & Configuration

In [5]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import re
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline               import Pipeline
from sklearn.preprocessing          import LabelEncoder, StandardScaler
from sklearn.model_selection        import (train_test_split, cross_val_score,
                                            StratifiedKFold, GridSearchCV)
from sklearn.svm                    import SVC, LinearSVC
from sklearn.linear_model           import (LogisticRegression, Ridge,
                                            RidgeClassifier)
from sklearn.ensemble               import RandomForestClassifier
from sklearn.multioutput            import MultiOutputRegressor
from sklearn.metrics                import (classification_report,
                                            confusion_matrix,
                                            ConfusionMatrixDisplay,
                                            f1_score, accuracy_score,
                                            mean_absolute_error, r2_score)
from sklearn.decomposition          import PCA
from scipy.sparse                   import hstack

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Config 
SEED        = 42
TEST_SIZE   = 0.20
CV_FOLDS    = 5
TFIDF_FEATS = 8000    # augmenté vs V1 (5000)

CLASS_NAMES_T1 = ['Excellent', 'Bon', 'Faible']
CLASS_NAMES_T3 = ['Positif', 'Neutre', 'Négatif']
CLASS_NAMES_T4 = ['Expert', 'Confirmé', 'Junior', 'Débutant']
CLASS_NAMES_T5 = ['Flash', 'Standard', 'Approfondie']
COLORS = {'Excellent':'#2ecc71','Bon':'#f39c12','Faible':'#e74c3c',
          'Expert':'#1a237e','Confirmé':'#1976D2','Junior':'#42A5F5','Débutant':'#90CAF9'}

np.random.seed(SEED)
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("  Imports OK — NLP Scoring V2")
print(f"   6 tâches : T1 qualité | T2 scores | T3 sentiment | T4 niveau | T5 format | T6 conformité")
print(f"   TF-IDF features : {TFIDF_FEATS}")


  Imports OK — NLP Scoring V2
   6 tâches : T1 qualité | T2 scores | T3 sentiment | T4 niveau | T5 format | T6 conformité
   TF-IDF features : 8000


# 2. Chargement des données — DS4 V2

In [9]:
df = pd.read_csv('conversation_transcripts_v2.csv', encoding='utf-8-sig')

print(f"Shape          : {df.shape}   (conversations × colonnes)")
print(f"Colonnes       : {list(df.columns)}")
print(f"\nDistributions :")
for col in ['response_quality','niveau_alia','visit_format','conformite_flag','client_sentiment']:
    print(f"  {col:<30}: {df[col].value_counts().to_dict()}")

print(f"\nScore stats :")
print(df[['scientific_accuracy_score','communication_clarity_score',
          'objection_handling_score','overall_score','acrv_score']].describe().round(2))

# Construire le texte d'entrée (objection + réponse)
df['input_text'] = df['objection_text'] + ' [SEP] ' + df['rep_response']
print(f"\n  input_text créé : objection + [SEP] + réponse")
print(f"   Longueur moyenne : {df['input_text'].str.len().mean():.0f} caractères")


Shape          : (600, 25)   (conversations × colonnes)
Colonnes       : ['conversation_id', 'rep_id', 'client_id', 'date', 'product', 'product_category', 'specialite_cible', 'objection_type', 'objection_text', 'rep_response', 'response_quality', 'niveau_alia', 'visit_format', 'visit_step', 'medecin_type', 'scientific_accuracy_score', 'communication_clarity_score', 'objection_handling_score', 'overall_score', 'acrv_score', 'conformite_flag', 'client_sentiment', 'conversation_duration_sec', 'outcome', 'skill_tags']

Distributions :
  response_quality              : {'Bon': 252, 'Excellent': 210, 'Faible': 138}
  niveau_alia                   : {'Junior': 237, 'Confirmé': 213, 'Expert': 86, 'Débutant': 64}
  visit_format                  : {'Standard': 276, 'Flash': 203, 'Approfondie': 121}
  conformite_flag               : {1: 462, 0: 138}
  client_sentiment              : {'Neutre': 285, 'Positif': 213, 'Négatif': 102}

Score stats :
       scientific_accuracy_score  communication_clar

# 3. Exploratory Data Analysis

## 3.1 Distributions des variables cibles

In [13]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('DS4 V2 — Distributions des variables cibles', fontsize=14, fontweight='bold')

# T1 - Qualité
ax = axes[0,0]
vals = df['response_quality'].value_counts()
bars = ax.bar(vals.index, vals.values,
              color=[COLORS[c] for c in vals.index], edgecolor='white', alpha=0.85)
ax.set_title('T1 — Qualité réponse', fontweight='bold')
for bar, v in zip(bars, vals.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, str(v),
            ha='center', fontweight='bold')

# T4 - Niveau ALIA
ax = axes[0,1]
vals4 = df['niveau_alia'].value_counts()
colors4 = [COLORS.get(c,'#888') for c in vals4.index]
bars4 = ax.bar(vals4.index, vals4.values, color=colors4, edgecolor='white', alpha=0.85)
ax.set_title('T4 — Niveau ALIA', fontweight='bold')
for bar, v in zip(bars4, vals4.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, str(v),
            ha='center', fontweight='bold')

# T5 - Format visite
ax = axes[0,2]
vals5 = df['visit_format'].value_counts()
ax.bar(vals5.index, vals5.values,
       color=['#5C6BC0','#26A69A','#EF5350'], edgecolor='white', alpha=0.85)
ax.set_title('T5 — Format visite', fontweight='bold')
for i,(v) in enumerate(vals5.values):
    ax.text(i, v+2, str(v), ha='center', fontweight='bold')

# T6 - Conformité
ax = axes[1,0]
vals6 = df['conformite_flag'].value_counts()
ax.bar(['Conforme (1)','Non-conforme (0)'], vals6.values,
       color=['#2ecc71','#e74c3c'], edgecolor='white', alpha=0.85)
ax.set_title('T6 — Conformité (mots tueurs)', fontweight='bold')
for i,v in enumerate(vals6.values):
    ax.text(i, v+2, str(v), ha='center', fontweight='bold')

# T3 - Sentiment
ax = axes[1,1]
vals3 = df['client_sentiment'].value_counts()
ax.bar(vals3.index, vals3.values,
       color=['#27ae60','#f39c12','#e74c3c'], edgecolor='white', alpha=0.85)
ax.set_title('T3 — Sentiment client', fontweight='bold')
for i,v in enumerate(vals3.values):
    ax.text(i, v+2, str(v), ha='center', fontweight='bold')

# Score distributions
ax = axes[1,2]
for q, color in COLORS.items():
    if q in ['Excellent','Bon','Faible']:
        subset = df[df['response_quality']==q]['overall_score']
        ax.hist(subset, bins=15, alpha=0.6, label=q, color=color, edgecolor='white')
ax.set_title('T2 — Overall Score par qualité', fontweight='bold')
ax.set_xlabel('Score ALIA (0–10)')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/nlp_01_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Distributions sauvegardées")

Distributions sauvegardées


## 3.2 Matrice qualité × niveau ALIA

In [16]:
pivot = pd.crosstab(df['response_quality'], df['niveau_alia'],
                    normalize='index').round(3) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Relation Qualité ↔ Niveau ALIA', fontsize=13, fontweight='bold')

# Heatmap
sns.heatmap(pivot, ax=axes[0], annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label':'%'})
axes[0].set_title('Distribution niveau par qualité (%)', fontweight='bold')

# Score moyen par niveau
score_by_niveau = df.groupby('niveau_alia')['overall_score'].agg(['mean','std'])
score_by_niveau = score_by_niveau.reindex(['Expert','Confirmé','Junior','Débutant'])
colors_n = [COLORS[n] for n in score_by_niveau.index]
axes[1].barh(score_by_niveau.index, score_by_niveau['mean'],
             xerr=score_by_niveau['std'], color=colors_n,
             alpha=0.85, edgecolor='white', capsize=4)
# Seuils référentiel officiel
for seuil, label, color in [(7.0,'Seuil Débutant→Junior','#f39c12'),
                             (8.0,'Seuil Junior→Confirmé','#e67e22'),
                             (9.0,'Seuil Confirmé→Expert','#e74c3c')]:
    axes[1].axvline(seuil, color=color, linestyle='--', alpha=0.7, label=label)
axes[1].set_xlabel('Score ALIA moyen (0–10)')
axes[1].set_title('Score moyen par niveau ALIA\n(seuils du référentiel officiel)',
                  fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_xlim(0, 10)

plt.tight_layout()
plt.savefig('outputs/nlp_02_qualite_niveau.png', dpi=120, bbox_inches='tight')
plt.show()

print(" Score moyen par niveau :")
print(score_by_niveau.round(2))
print("\n Seuils officiels (Référentiel ALIA) :")
print("   Débutant → Junior   : overall_score >= 7.0")
print("   Junior   → Confirmé : overall_score >= 8.0")
print("   Confirmé → Expert   : overall_score >= 9.0")


 Score moyen par niveau :
             mean   std
niveau_alia            
Expert       8.48  0.79
Confirmé     7.63  1.38
Junior       5.62  1.57
Débutant     3.82  0.70

 Seuils officiels (Référentiel ALIA) :
   Débutant → Junior   : overall_score >= 7.0
   Junior   → Confirmé : overall_score >= 8.0
   Confirmé → Expert   : overall_score >= 9.0


## 3.3 Analyse ACRV et conformité

In [19]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Analyse A-C-R-V & Conformité par qualité', fontsize=13, fontweight='bold')

# ACRV score par qualité
ax = axes[0]
for quality, color in [('Excellent','#2ecc71'),('Bon','#f39c12'),('Faible','#e74c3c')]:
    subset = df[df['response_quality']==quality]['acrv_score']
    ax.hist(subset, bins=5, range=(-0.5,4.5), alpha=0.65,
            label=quality, color=color, edgecolor='white')
ax.set_xticks([0,1,2,3,4])
ax.set_xticklabels(['0\n(aucune)','1\n(A)','2\n(A-C)','3\n(A-C-R)','4\n(complet)'])
ax.set_title('Score A-C-R-V par qualité', fontweight='bold')
ax.set_xlabel('Étapes A-C-R-V complétées')
ax.legend()

# Conformité par qualité
ax = axes[1]
conf_by_q = df.groupby(['response_quality','conformite_flag']).size().unstack(fill_value=0)
conf_by_q.plot(kind='bar', ax=ax, color=['#e74c3c','#2ecc71'],
               edgecolor='white', alpha=0.85, width=0.6)
ax.set_title('Conformité par qualité', fontweight='bold')
ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(['Non-conforme (0)','Conforme (1)'])

# ACRV moyen par niveau ALIA
ax = axes[2]
acrv_niveau = df.groupby('niveau_alia')['acrv_score'].mean().reindex(
    ['Débutant','Junior','Confirmé','Expert'])
colors_n = [COLORS[n] for n in acrv_niveau.index]
bars = ax.bar(acrv_niveau.index, acrv_niveau.values,
              color=colors_n, edgecolor='white', alpha=0.85)
ax.axhline(4, color='green', linestyle='--', alpha=0.5, label='Méthode complète')
ax.set_ylim(0, 4.5)
ax.set_title('Score A-C-R-V moyen par niveau ALIA', fontweight='bold')
for bar, v in zip(bars, acrv_niveau.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f'{v:.2f}', ha='center', fontweight='bold', fontsize=10)
ax.legend()

plt.tight_layout()
plt.savefig('outputs/nlp_03_acrv_conformite.png', dpi=120, bbox_inches='tight')
plt.show()

print(" ACRV moyen par niveau :", acrv_niveau.round(3).to_dict())
print(" Expert : ACRV ≈ 3.7/4 | Débutant : ACRV ≈ 0/4")


 ACRV moyen par niveau : {'Débutant': 0.078, 'Junior': 1.561, 'Confirmé': 2.944, 'Expert': 3.884}
 Expert : ACRV ≈ 3.7/4 | Débutant : ACRV ≈ 0/4


## 3.4 Produits et formats de visite

In [22]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Analyse par produit et format de visite', fontsize=13, fontweight='bold')

# Score par produit
prod_scores = df.groupby('product')['overall_score'].mean().sort_values(ascending=True)
colors_p = ['#e74c3c' if v < 6 else '#f39c12' if v < 7.5 else '#2ecc71'
            for v in prod_scores.values]
axes[0].barh(prod_scores.index, prod_scores.values, color=colors_p,
             alpha=0.85, edgecolor='white')
axes[0].axvline(7.0, color='orange', linestyle='--', alpha=0.7, label='Seuil 7.0')
axes[0].set_title('Score ALIA moyen par produit', fontweight='bold')
axes[0].set_xlabel('Overall Score moyen')
axes[0].legend()

# ACRV par format visite
acrv_format = df.groupby('visit_format')['acrv_score'].value_counts(normalize=True).unstack()
acrv_format.plot(kind='bar', ax=axes[1], colormap='Blues', edgecolor='white',
                 alpha=0.85, width=0.6)
axes[1].set_title('Distribution ACRV par format de visite', fontweight='bold')
axes[1].set_xlabel('Format de visite')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='ACRV score', bbox_to_anchor=(1.05,1))

plt.tight_layout()
plt.savefig('outputs/nlp_04_produit_format.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. NLPFeatureExtractor V2 — 24 features linguistiques

Les features sont extraites directement depuis le texte des réponses.
Elles sont inspirées des **6 étapes de la visite médicale** (Manuel VITAL),
de la méthode **A-C-R-V** (objections) et du **Référentiel 4 niveaux**.


In [25]:
class NLPFeatureExtractorV2:
    """
    Extracteur de 24 features linguistiques interprétables.
    
    Basé sur :
    - Manuel VITAL SA — 6 étapes de la visite médicale
    - Méthode A-C-R-V (Accueillir-Clarifier-Répondre-Valider)
    - Référentiel des 4 niveaux de compétence ALIA
    - Annexe V2 Scripts Top Sellers VITAL
    """

    # Patterns linguistiques VITAL

    # Étape 1 — Permission / Introduction (Manuel section 4.1)
    PERMISSION = [r"\bc'est ok\b", r"\b2 minutes\b", r"\btrès court\b",
                  r"\bpermission\b", r"\bbonjour docteur\b", r"\bje fais court\b"]

    # Étape 2 — Sondage / Questions (Manuel section 4.2)
    QUESTIONS  = [r"\?", r"\bc'est plutôt\b", r"\bvous voyez\b",
                  r"\bquel\b", r"\bcomment\b", r"\bpourquoi\b",
                  r"\bsur quels\b", r"\bvotre critère\b"]

    # Étape 3 — Synthèse (Manuel section 4.3)
    SYNTHESE   = [r"\bsi je résume\b", r"\bvotre priorité\b",
                  r"\bvotre attente\b", r"\bc'est bien ça\b",
                  r"\bsi je comprends\b", r"\breformul\b"]

    # Étape 4 — A-C-R-V (Manuel section 4.4)
    ACRV_A = [r"\bje comprends\b", r"\bc'est légitime\b", r"\bje vous remercie\b",
              r"\bpertinent\b", r"\bvalable\b", r"\bc'est normal\b"]
    ACRV_C = [r"\bquand vous dites\b", r"\bc'est plutôt\b", r"\bvous pensez à\b",
              r"\bpour bien\b", r"\bvous craignez\b", r"\bvous attendez\b"]
    ACRV_R = [r"\bselon\b", r"\ben pratique\b", r"\bprofil\b",
              r"\btest\b", r"\bdonnées\b", r"\bfiche\b", r"\brepère\b",
              r"\bconseils\b", r"\bplan b\b"]
    ACRV_V = [r"\best-ce que\b", r"\brépond\b", r"\bd'accord\b",
              r"\bpensez-vous\b", r"\bseriez-vous\b", r"\bacceptable\b",
              r"\bça vous semble\b"]

    # Étape 5 — Argumentation (Manuel section 4.5)
    PREUVE    = [r"\bétude\b", r"\bclinique\b", r"\bdonnées\b",
                 r"\bselon la notice\b", r"\brepère\b", r"\bpreuve\b"]
    PROFIL_PT = [r"\bchez ce profil\b", r"\bpour ce type\b", r"\bces patients\b",
                 r"\bprofil ciblé\b", r"\bchez vos patients\b", r"\bsur ces cas\b"]
    AVANTAGE  = [r"\bbénéfice\b", r"\bavantage\b", r"\bsimple\b",
                 r"\bobservance\b", r"\bcure\b", r"\broutine\b"]

    # Étape 6 — BIP / Closing (Manuel section 4.6)
    BIP_SIGNAL = [r"\bessayer\b", r"\b2.?3 patients\b", r"\bje repasse\b",
                  r"\bon est d'accord\b", r"\bengagement\b", r"\btest sur\b",
                  r"\bje reviens\b", r"\bplan de test\b"]

    # Conformité (Référentiel 4 niveaux + Annexe V2)
    CONFORMITE = [r"\bselon la notice\b", r"\bje vérifie\b", r"\bje reviens avec\b",
                  r"\bdans le cadre\b", r"\bselon les données\b",
                  r"\bje vous confirme par\b", r"\bselon la fiche\b"]

    # Mots tueurs / surpromesses (Référentiel — KPI Confirmé/Expert)
    MOTS_TUEURS = [r"\bgaranti\b", r"\b100%\b", r"\bguérit\b",
                   r"\btoujours efficace\b", r"\baucun effet secondaire\b",
                   r"\bmeilleur produit\b", r"\brévolutionnaire\b",
                   r"\bmiracle\b", r"\bsans aucun risque\b",
                   r"\bprouvé sans réserve\b"]

    # Empathie générale
    EMPATHIE   = [r"\bje comprends\b", r"\bvous avez raison\b",
                  r"\bc'est une bonne question\b", r"\bje vous entends\b"]

    # Vague / non-structuré
    VAGUE      = [r"\bc'est bien\b", r"\bpas de problème\b",
                  r"\bça marche\b", r"\bne vous inquiétez pas\b"]

    def _match(self, text, patterns):
        t = text.lower()
        return int(any(re.search(p, t) for p in patterns))

    def _count_matches(self, text, patterns):
        t = text.lower()
        return sum(1 for p in patterns if re.search(p, t))

    def extract(self, objection: str, response: str) -> dict:
        """
        Extrait les 24 features depuis une paire (objection, réponse).
        
        Returns:
            dict : 24 features linguistiques nommées
        """
        r = response
        o = objection

        # A-C-R-V (4 features) 
        has_acrv_a = self._match(r, self.ACRV_A)       # Accueil
        has_acrv_c = self._match(r, self.ACRV_C)       # Clarification
        has_acrv_r = self._match(r, self.ACRV_R)       # Réponse
        has_acrv_v = self._match(r, self.ACRV_V)       # Validation
        acrv_score = has_acrv_a + has_acrv_c + has_acrv_r + has_acrv_v

        # Étapes visite (4 features) 
        has_permission = self._match(r+o, self.PERMISSION)
        has_synthese   = self._match(r,   self.SYNTHESE)
        has_preuve     = self._match(r,   self.PREUVE)
        has_bip        = self._match(r,   self.BIP_SIGNAL)

        # Argumentation (3 features) 
        has_profil_patient = self._match(r, self.PROFIL_PT)
        has_avantage       = self._match(r, self.AVANTAGE)
        has_decouverte     = self._match(o, self.QUESTIONS)  # questions dans objection

        # Conformité (3 features) 
        has_conformite     = self._match(r, self.CONFORMITE)
        has_mot_tueur      = self._match(r, self.MOTS_TUEURS)
        n_mots_tueurs      = self._count_matches(r, self.MOTS_TUEURS)

        # Structure générale (6 features) 
        has_empathie       = self._match(r, self.EMPATHIE)
        is_vague           = self._match(r, self.VAGUE)
        question_count     = len(re.findall(r'\?', r))
        word_count         = len(r.split())
        sentence_count     = len(re.split(r'[.!?]+', r))
        has_numbers        = int(bool(re.search(r'\d+', r)))

        # Scores composites (4 features) 
        argumentation_richness = (has_preuve + has_profil_patient +
                                   has_avantage + has_decouverte) / 4.0
        conformite_score       = max(0, has_conformite - n_mots_tueurs * 0.5)
        response_quality_proxy = (acrv_score / 4.0 * 0.4 +
                                   has_bip * 0.2 +
                                   has_preuve * 0.2 +
                                   (1 - is_vague) * 0.2)
        structure_score        = (has_permission + has_synthese + has_preuve +
                                   has_bip) / 4.0

        return {
            # A-C-R-V
            'has_acrv_a'             : has_acrv_a,
            'has_acrv_c'             : has_acrv_c,
            'has_acrv_r'             : has_acrv_r,
            'has_acrv_v'             : has_acrv_v,
            'acrv_completeness'      : acrv_score,
            # Étapes visite
            'has_permission'         : has_permission,
            'has_synthese'           : has_synthese,
            'has_preuve'             : has_preuve,
            'has_bip_signal'         : has_bip,
            # Argumentation
            'has_profil_patient'     : has_profil_patient,
            'has_avantage'           : has_avantage,
            'has_decouverte'         : has_decouverte,
            # Conformité
            'has_conformite'         : has_conformite,
            'has_mot_tueur'          : has_mot_tueur,
            'n_mots_tueurs'          : n_mots_tueurs,
            # Structure
            'has_empathie'           : has_empathie,
            'is_vague'               : is_vague,
            'question_count'         : question_count,
            'word_count'             : word_count,
            'sentence_count'         : sentence_count,
            'has_numbers'            : has_numbers,
            # Scores composites
            'argumentation_richness' : round(argumentation_richness, 3),
            'conformite_score'       : round(conformite_score, 3),
            'response_quality_proxy' : round(response_quality_proxy, 3),
        }

    def extract_batch(self, df):
        """Extrait les features sur tout le dataframe."""
        feats = [self.extract(row['objection_text'], row['rep_response'])
                 for _, row in df.iterrows()]
        return pd.DataFrame(feats)

# Extraire et afficher 
extractor = NLPFeatureExtractorV2()
df_feats  = extractor.extract_batch(df)

print(f" 24 features extraites : {df_feats.shape}")
print(f"\nFeatures :")
for i, col in enumerate(df_feats.columns, 1):
    print(f"  {i:>2}. {col}")


 24 features extraites : (600, 24)

Features :
   1. has_acrv_a
   2. has_acrv_c
   3. has_acrv_r
   4. has_acrv_v
   5. acrv_completeness
   6. has_permission
   7. has_synthese
   8. has_preuve
   9. has_bip_signal
  10. has_profil_patient
  11. has_avantage
  12. has_decouverte
  13. has_conformite
  14. has_mot_tueur
  15. n_mots_tueurs
  16. has_empathie
  17. is_vague
  18. question_count
  19. word_count
  20. sentence_count
  21. has_numbers
  22. argumentation_richness
  23. conformite_score
  24. response_quality_proxy


In [27]:
# Visualiser les features par qualité 
fig, axes = plt.subplots(4, 6, figsize=(22, 14))
fig.suptitle('24 Features NLP par classe de qualité (T1)', fontsize=13, fontweight='bold')
axes = axes.flatten()

df_viz = pd.concat([df[['response_quality']], df_feats], axis=1)

for idx, feat in enumerate(df_feats.columns):
    ax = axes[idx]
    for quality, color in [('Excellent','#2ecc71'),('Bon','#f39c12'),('Faible','#e74c3c')]:
        vals = df_viz[df_viz['response_quality']==quality][feat]
        ax.bar(quality, vals.mean(), color=color, alpha=0.8, edgecolor='white',
               yerr=vals.std(), capsize=3, width=0.6)
    ax.set_title(feat, fontsize=7, fontweight='bold')
    ax.tick_params(axis='x', labelsize=6)
    ax.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig('outputs/nlp_05_features_by_quality.png', dpi=110, bbox_inches='tight')
plt.show()

# Top features discriminantes
print("\n Features les plus discriminantes (écart Excellent vs Faible) :")
disc = {}
for feat in df_feats.columns:
    mean_ex = df_viz[df_viz['response_quality']=='Excellent'][feat].mean()
    mean_fa = df_viz[df_viz['response_quality']=='Faible'][feat].mean()
    disc[feat] = abs(mean_ex - mean_fa)
disc_sorted = sorted(disc.items(), key=lambda x: x[1], reverse=True)
for feat, val in disc_sorted[:10]:
    print(f"  {feat:<30} : Δ={val:.3f}")



 Features les plus discriminantes (écart Excellent vs Faible) :
  word_count                     : Δ=25.124
  acrv_completeness              : Δ=2.140
  question_count                 : Δ=2.000
  sentence_count                 : Δ=1.362
  n_mots_tueurs                  : Δ=1.022
  has_acrv_v                     : Δ=0.971
  has_mot_tueur                  : Δ=0.768
  has_acrv_r                     : Δ=0.578
  has_conformite                 : Δ=0.519
  conformite_score               : Δ=0.519


## 5. Préparation des features — TF-IDF + Features linguistiques

In [30]:
from scipy.sparse import hstack, csr_matrix

# TF-IDF 
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=TFIDF_FEATS,
    min_df=2,
    sublinear_tf=True,
    analyzer='word',
    strip_accents=None,   # conserver les accents français
)
X_tfidf = tfidf.fit_transform(df['input_text'])

# Features linguistiques (sparse) 
scaler_feats = StandardScaler()
X_linguistic = csr_matrix(
    scaler_feats.fit_transform(df_feats.values)
)

# Combinaison TF-IDF + features linguistiques 
X_combined = hstack([X_tfidf, X_linguistic])

print(f"X_tfidf      : {X_tfidf.shape}")
print(f"X_linguistic : {X_linguistic.shape}")
print(f"X_combined   : {X_combined.shape}")

# Encodage des labels 
le_t1 = LabelEncoder()
le_t3 = LabelEncoder()
le_t4 = LabelEncoder()
le_t5 = LabelEncoder()

y_t1 = le_t1.fit_transform(df['response_quality'])
y_t2 = df[['scientific_accuracy_score',
           'communication_clarity_score',
           'objection_handling_score']].values
y_t3 = le_t3.fit_transform(df['client_sentiment'])
y_t4 = le_t4.fit_transform(df['niveau_alia'])
y_t5 = le_t5.fit_transform(df['visit_format'])
y_t6 = df['conformite_flag'].values

print(f"\nLabels encodés :")
print(f"  T1 classes : {list(le_t1.classes_)}")
print(f"  T3 classes : {list(le_t3.classes_)}")
print(f"  T4 classes : {list(le_t4.classes_)}")
print(f"  T5 classes : {list(le_t5.classes_)}")
print(f"  T6 (binaire) : [0, 1]")

# Train/Test split
X_tr, X_te, idx_tr, idx_te = train_test_split(
    X_combined, np.arange(len(df)),
    test_size=TEST_SIZE, random_state=SEED, stratify=y_t1)

# Labels splits
y_t1_tr, y_t1_te = y_t1[idx_tr], y_t1[idx_te]
y_t2_tr, y_t2_te = y_t2[idx_tr], y_t2[idx_te]
y_t3_tr, y_t3_te = y_t3[idx_tr], y_t3[idx_te]
y_t4_tr, y_t4_te = y_t4[idx_tr], y_t4[idx_te]
y_t5_tr, y_t5_te = y_t5[idx_tr], y_t5[idx_te]
y_t6_tr, y_t6_te = y_t6[idx_tr], y_t6[idx_te]

print(f"\nSplit : train={X_tr.shape[0]} | test={X_te.shape[0]}")

# Sauvegarder pour nlp_scoring_train.py
joblib.dump(tfidf,         'models/tfidf_v2.pkl')
joblib.dump(scaler_feats,  'models/scaler_feats_v2.pkl')
joblib.dump(extractor,     'models/feature_extractor_v2.pkl')
joblib.dump({'le_t1':le_t1,'le_t3':le_t3,'le_t4':le_t4,'le_t5':le_t5},
            'models/label_encoders_v2.pkl')
print(" Artifacts TF-IDF + encoders sauvegardés")


X_tfidf      : (600, 1525)
X_linguistic : (600, 24)
X_combined   : (600, 1549)

Labels encodés :
  T1 classes : ['Bon', 'Excellent', 'Faible']
  T3 classes : ['Neutre', 'Négatif', 'Positif']
  T4 classes : ['Confirmé', 'Débutant', 'Expert', 'Junior']
  T5 classes : ['Approfondie', 'Flash', 'Standard']
  T6 (binaire) : [0, 1]

Split : train=480 | test=120
 Artifacts TF-IDF + encoders sauvegardés


## 6. Benchmarks — Baseline models T1 → T6

## 6.1 T1 — Classification qualité (SVM RBF)

In [34]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

# T1 : SVM RBF (modèle retenu V1) 
svm_t1 = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=SEED)
cv_t1  = cross_val_score(svm_t1, X_tr, y_t1_tr, cv=cv, scoring='f1_macro')
svm_t1.fit(X_tr, y_t1_tr)
y_pred_t1 = svm_t1.predict(X_te)
f1_t1     = f1_score(y_te:=y_t1_te, y_pred_t1, average='macro')

print("=" * 60)
print("  T1 — Classification Qualité — SVM RBF")
print("=" * 60)
print(f"  CV F1-Macro  : {cv_t1.mean():.4f} ± {cv_t1.std():.4f}")
print(f"  Test F1-Macro: {f1_t1:.4f}")
print(f"  Test Accuracy: {accuracy_score(y_t1_te, y_pred_t1):.4f}")
print()
print(classification_report(y_t1_te, y_pred_t1,
      target_names=le_t1.classes_, digits=4))

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(y_t1_te, y_pred_t1),
                       display_labels=le_t1.classes_).plot(ax=ax, cmap='Blues')
ax.set_title(f'T1 — Qualité (F1={f1_t1:.4f})', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/nlp_06_cm_t1.png', dpi=110, bbox_inches='tight')
plt.show()

joblib.dump(svm_t1, 'models/t1_quality_v2.pkl')


  T1 — Classification Qualité — SVM RBF
  CV F1-Macro  : 0.9982 ± 0.0036
  Test F1-Macro: 1.0000
  Test Accuracy: 1.0000

              precision    recall  f1-score   support

         Bon     1.0000    1.0000    1.0000        50
   Excellent     1.0000    1.0000    1.0000        42
      Faible     1.0000    1.0000    1.0000        28

    accuracy                         1.0000       120
   macro avg     1.0000    1.0000    1.0000       120
weighted avg     1.0000    1.0000    1.0000       120



['models/t1_quality_v2.pkl']

## 6.2 T2 — Régression scores (Ridge MultiOutput)

In [37]:
# T2 : Ridge MultiOutputRegressor 
ridge_t2 = MultiOutputRegressor(Ridge(alpha=1.0))
ridge_t2.fit(X_tr, y_t2_tr)
y_pred_t2 = ridge_t2.predict(X_te)

score_names = ['scientific_accuracy', 'communication_clarity', 'objection_handling']
print("=" * 60)
print("  T2 — Régression Scores — Ridge MultiOutput")
print("=" * 60)
maes, r2s = [], []
for i, name in enumerate(score_names):
    mae = mean_absolute_error(y_t2_te[:,i], y_pred_t2[:,i])
    r2  = r2_score(y_t2_te[:,i], y_pred_t2[:,i])
    maes.append(mae); r2s.append(r2)
    print(f"  {name:<25} MAE={mae:.4f}  R²={r2:.4f}")

# Score pondéré ALIA
w = np.array([0.25, 0.30, 0.45])
overall_pred = (y_pred_t2 * w).sum(axis=1)
overall_true = (y_t2_te  * w).sum(axis=1)
mae_ov = mean_absolute_error(overall_true, overall_pred)
r2_ov  = r2_score(overall_true, overall_pred)
print(f"\n  Overall Score (ALIA)      MAE={mae_ov:.4f}  R²={r2_ov:.4f}")
print(f"  Formule : 0.25×sci + 0.30×clar + 0.45×obj")

# Scatter plot
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('T2 — Prédiction scores (Ridge MultiOutput)', fontweight='bold')
for i, (name, ax) in enumerate(zip(score_names + ['Overall (ALIA)'],
                                    axes.flatten())):
    if i < 3:
        true_v = y_t2_te[:,i]; pred_v = y_pred_t2[:,i]
    else:
        true_v = overall_true; pred_v = overall_pred
    ax.scatter(true_v, pred_v, alpha=0.4, s=15, color='#3498db')
    ax.plot([0,10],[0,10],'r--', alpha=0.5)
    ax.set_xlabel('Vrai score'); ax.set_ylabel('Score prédit')
    mae_i = mean_absolute_error(true_v, pred_v)
    r2_i  = r2_score(true_v, pred_v)
    ax.set_title(f'{name}\nMAE={mae_i:.3f} R²={r2_i:.3f}', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/nlp_07_regression_t2.png', dpi=110, bbox_inches='tight')
plt.show()

joblib.dump(ridge_t2, 'models/t2_scores_v2.pkl')


  T2 — Régression Scores — Ridge MultiOutput
  scientific_accuracy       MAE=1.0106  R²=0.6188
  communication_clarity     MAE=0.9958  R²=0.6789
  objection_handling        MAE=0.9026  R²=0.6531

  Overall Score (ALIA)      MAE=0.6962  R²=0.7567
  Formule : 0.25×sci + 0.30×clar + 0.45×obj


['models/t2_scores_v2.pkl']

## 6.3 T3 — Sentiment client

In [40]:
lr_t3  = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
cv_t3  = cross_val_score(lr_t3, X_tr, y_t3_tr, cv=cv, scoring='f1_macro')
lr_t3.fit(X_tr, y_t3_tr)
y_pred_t3 = lr_t3.predict(X_te)
f1_t3     = f1_score(y_t3_te, y_pred_t3, average='macro')

print("=" * 60)
print("  T3 — Sentiment Client — Logistic Regression")
print("=" * 60)
print(f"  CV F1-Macro  : {cv_t3.mean():.4f} ± {cv_t3.std():.4f}")
print(f"  Test F1-Macro: {f1_t3:.4f}")
print(classification_report(y_t3_te, y_pred_t3,
      target_names=le_t3.classes_, digits=4))

joblib.dump(lr_t3, 'models/t3_sentiment_v2.pkl')


  T3 — Sentiment Client — Logistic Regression
  CV F1-Macro  : 0.7571 ± 0.0362
  Test F1-Macro: 0.6901
              precision    recall  f1-score   support

      Neutre     0.6071    0.6415    0.6239        53
     Négatif     0.7692    0.9524    0.8511        21
     Positif     0.6579    0.5435    0.5952        46

    accuracy                         0.6583       120
   macro avg     0.6781    0.7125    0.6901       120
weighted avg     0.6550    0.6583    0.6526       120



['models/t3_sentiment_v2.pkl']

## 6.4 T4 — Niveau ALIA (Débutant/Junior/Confirmé/Expert)

In [43]:
svm_t4 = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=SEED)
cv_t4  = cross_val_score(svm_t4, X_tr, y_t4_tr, cv=cv, scoring='f1_macro')
svm_t4.fit(X_tr, y_t4_tr)
y_pred_t4 = svm_t4.predict(X_te)
f1_t4     = f1_score(y_t4_te, y_pred_t4, average='macro')

print("=" * 60)
print("  T4 — Niveau ALIA — SVM RBF")
print("=" * 60)
print(f"  CV F1-Macro  : {cv_t4.mean():.4f} ± {cv_t4.std():.4f}")
print(f"  Test F1-Macro: {f1_t4:.4f}")
print(f"  Test Accuracy: {accuracy_score(y_t4_te, y_pred_t4):.4f}")
print()
print(classification_report(y_t4_te, y_pred_t4,
      target_names=le_t4.classes_, digits=4))

fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(confusion_matrix(y_t4_te, y_pred_t4),
                       display_labels=le_t4.classes_).plot(ax=ax, cmap='Blues')
ax.set_title(f'T4 — Niveau ALIA (F1={f1_t4:.4f})', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/nlp_08_cm_t4.png', dpi=110, bbox_inches='tight')
plt.show()

joblib.dump(svm_t4, 'models/t4_niveau_v2.pkl')


  T4 — Niveau ALIA — SVM RBF
  CV F1-Macro  : 0.5360 ± 0.0451
  Test F1-Macro: 0.6025
  Test Accuracy: 0.7250

              precision    recall  f1-score   support

    Confirmé     0.7105    0.6429    0.6750        42
    Débutant     0.7500    0.3333    0.4615         9
      Expert     0.6250    0.3333    0.4348        15
      Junior     0.7429    0.9630    0.8387        54

    accuracy                         0.7250       120
   macro avg     0.7071    0.5681    0.6025       120
weighted avg     0.7173    0.7250    0.7026       120



['models/t4_niveau_v2.pkl']

## 6.5 T5 — Format visite (Flash/Standard/Approfondie)

In [46]:
lr_t5  = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
cv_t5  = cross_val_score(lr_t5, X_tr, y_t5_tr, cv=cv, scoring='f1_macro')
lr_t5.fit(X_tr, y_t5_tr)
y_pred_t5 = lr_t5.predict(X_te)
f1_t5     = f1_score(y_t5_te, y_pred_t5, average='macro')

print("=" * 60)
print("  T5 — Format Visite — Logistic Regression")
print("=" * 60)
print(f"  CV F1-Macro  : {cv_t5.mean():.4f} ± {cv_t5.std():.4f}")
print(f"  Test F1-Macro: {f1_t5:.4f}")
print(classification_report(y_t5_te, y_pred_t5,
      target_names=le_t5.classes_, digits=4))

joblib.dump(lr_t5, 'models/t5_format_v2.pkl')


  T5 — Format Visite — Logistic Regression
  CV F1-Macro  : 0.7699 ± 0.0562
  Test F1-Macro: 0.7576
              precision    recall  f1-score   support

 Approfondie     0.5484    0.7391    0.6296        23
       Flash     0.9697    0.8205    0.8889        39
    Standard     0.7679    0.7414    0.7544        58

    accuracy                         0.7667       120
   macro avg     0.7620    0.7670    0.7576       120
weighted avg     0.7914    0.7667    0.7742       120



['models/t5_format_v2.pkl']

## 6.6 T6 — Conformité / Détection mots tueurs

In [49]:
lr_t6  = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
cv_t6  = cross_val_score(lr_t6, X_tr, y_t6_tr, cv=cv, scoring='f1_macro')
lr_t6.fit(X_tr, y_t6_tr)
y_pred_t6 = lr_t6.predict(X_te)
f1_t6     = f1_score(y_t6_te, y_pred_t6, average='macro')

print("=" * 60)
print("  T6 — Conformité — Logistic Regression")
print("=" * 60)
print(f"  CV F1-Macro  : {cv_t6.mean():.4f} ± {cv_t6.std():.4f}")
print(f"  Test F1-Macro: {f1_t6:.4f}")
print(f"  Test Accuracy: {accuracy_score(y_t6_te, y_pred_t6):.4f}")
print(classification_report(y_t6_te, y_pred_t6,
      target_names=['Non-conforme','Conforme'], digits=4))

# Note : T6 peut aussi être une règle déterministe (regex mots tueurs)
print("\n  T6 peut être résolu par règle déterministe (regex) :")
print("    conformite = not any(mot_tueur in response for mot_tueur in MOTS_TUEURS)")
print("    -> Précision parfaite sur données synthétiques")

joblib.dump(lr_t6, 'models/t6_conformite_v2.pkl')


  T6 — Conformité — Logistic Regression
  CV F1-Macro  : 1.0000 ± 0.0000
  Test F1-Macro: 1.0000
  Test Accuracy: 1.0000
              precision    recall  f1-score   support

Non-conforme     1.0000    1.0000    1.0000        28
    Conforme     1.0000    1.0000    1.0000        92

    accuracy                         1.0000       120
   macro avg     1.0000    1.0000    1.0000       120
weighted avg     1.0000    1.0000    1.0000       120


  T6 peut être résolu par règle déterministe (regex) :
    conformite = not any(mot_tueur in response for mot_tueur in MOTS_TUEURS)
    -> Précision parfaite sur données synthétiques


['models/t6_conformite_v2.pkl']

# 7. Benchmark global — Récapitulatif T1→T6

In [52]:
print("=" * 70)
print("  BENCHMARK GLOBAL — NLP Scoring V2 — 6 tâches")
print("=" * 70)
print(f"  {'Tâche':<8}  {'Modèle':<25}  {'CV F1':>8}  {'Test F1':>9}  {'Note'}")
print("  " + "-" * 66)
results = [
    ("T1", "SVM RBF",        cv_t1.mean(), f1_t1, "Qualité réponse"),
    ("T2", "Ridge MultiOut", None,          r2_ov, "R² overall score"),
    ("T3", "LogReg",         cv_t3.mean(), f1_t3, "Sentiment client"),
    ("T4", "SVM RBF",        cv_t4.mean(), f1_t4, "Niveau ALIA"),
    ("T5", "LogReg",         cv_t5.mean(), f1_t5, "Format visite"),
    ("T6", "LogReg + Règle", cv_t6.mean(), f1_t6, "Conformité"),
]
for task, model, cv_f, te_f, note in results:
    cv_s = f"{cv_f:.4f}" if cv_f else "  —   "
    print(f"  {task:<8}  {model:<25}  {cv_s:>8}  {te_f:>9.4f}  {note}")
print("=" * 70)

# Visualisation radar / bar
fig, ax = plt.subplots(figsize=(11, 5))
tasks  = ['T1\nQualité','T2\nR² overall','T3\nSentiment','T4\nNiveau ALIA','T5\nFormat','T6\nConformité']
scores = [f1_t1, r2_ov, f1_t3, f1_t4, f1_t5, f1_t6]
colors_b = ['#2ecc71' if s >= 0.85 else '#f39c12' if s >= 0.65 else '#e74c3c' for s in scores]
bars = ax.bar(tasks, scores, color=colors_b, edgecolor='white', alpha=0.85, width=0.5)
ax.axhline(0.85, color='green',  linestyle='--', alpha=0.5, label='Cible ≥ 0.85')
ax.axhline(0.65, color='orange', linestyle='--', alpha=0.5, label='Acceptable ≥ 0.65')
for bar, s in zip(bars, scores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{s:.3f}', ha='center', fontweight='bold', fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title('Benchmark NLP Scoring V2 — 6 tâches', fontsize=13, fontweight='bold')
ax.set_ylabel('F1-Macro / R²')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/nlp_09_benchmark.png', dpi=120, bbox_inches='tight')
plt.show()


  BENCHMARK GLOBAL — NLP Scoring V2 — 6 tâches
  Tâche     Modèle                        CV F1    Test F1  Note
  ------------------------------------------------------------------
  T1        SVM RBF                      0.9982     1.0000  Qualité réponse
  T2        Ridge MultiOut                 —        0.7567  R² overall score
  T3        LogReg                       0.7571     0.6901  Sentiment client
  T4        SVM RBF                      0.5360     0.6025  Niveau ALIA
  T5        LogReg                       0.7699     0.7576  Format visite
  T6        LogReg + Règle               1.0000     1.0000  Conformité


# 8. Démo inférence — 3 cas test

In [55]:
def predict_full(objection, response):
    """Pipeline d'inférence complète sur une conversation."""
    text = objection + ' [SEP] ' + response
    feats = extractor.extract(objection, response)
    feat_arr = scaler_feats.transform([list(feats.values())])
    
    from scipy.sparse import csr_matrix, hstack as sp_hstack
    x_tf   = tfidf.transform([text])
    x_feat = csr_matrix(feat_arr)
    x_comb = sp_hstack([x_tf, x_feat])
    
    # T1 — Qualité
    q_pred  = le_t1.inverse_transform(svm_t1.predict(x_comb))[0]
    q_proba = dict(zip(le_t1.classes_, svm_t1.predict_proba(x_comb)[0].round(3)))
    
    # T2 — Scores
    scores_pred = ridge_t2.predict(x_comb)[0]
    sci, clar, obj_s = scores_pred
    overall = 0.25*sci + 0.30*clar + 0.45*obj_s
    
    # T3 — Sentiment
    sent = le_t3.inverse_transform(lr_t3.predict(x_comb))[0]
    
    # T4 — Niveau ALIA
    niveau = le_t4.inverse_transform(svm_t4.predict(x_comb))[0]
    
    # T5 — Format visite
    vformat = le_t5.inverse_transform(lr_t5.predict(x_comb))[0]
    
    # T6 — Conformité (règle + modèle)
    conf_regle  = int(not any(re.search(p, response.lower()) for p in
                  NLPFeatureExtractorV2.MOTS_TUEURS))
    
    # Niveau seuil passage
    seuils = {'Débutant': 7.0, 'Junior': 8.0, 'Confirmé': 9.0, 'Expert': 10.0}
    seuil  = seuils.get(niveau, 7.0)
    next_n = {'Débutant':'Junior','Junior':'Confirmé','Confirmé':'Expert','Expert':'—'}
    
    return {
        'quality'        : q_pred,
        'quality_proba'  : q_proba,
        'scores'         : {'scientific': round(sci,2), 'clarity': round(clar,2), 'objection': round(obj_s,2)},
        'overall_score'  : round(overall, 2),
        'sentiment'      : sent,
        'niveau_alia'    : niveau,
        'visit_format'   : vformat,
        'conformite'     : bool(conf_regle),
        'acrv_score'     : feats['acrv_completeness'],
        'seuil_passage'  : seuil,
        'prochain_niveau': next_n[niveau],
    }

# Test cases 
test_cases = [
    ("Pas convaincu par ce produit",
     "Je comprends tout à fait votre préoccupation. Quand vous dites pas convaincu, "
     "c'est plutôt sur la composition ou la tolérance ? Selon les données disponibles, "
     "1 repère ciblé et proposition de test sur 2 à 3 patients. "
     "Est-ce que ce repère suffit pour envisager un test ? (selon la notice)"),
    ("J'ai mes habitudes avec un autre produit",
     "Je comprends. Votre remarque est pertinente. Sur ce type de patients, "
     "on peut positionner en plan B sur les cas moins satisfaits. Je vous laisse une fiche."),
    ("Pas le temps pour une présentation",
     "C'est le meilleur produit de notre gamme. Je vous garantis que vos patients "
     "seront toujours satisfaits. Aucun effet secondaire signalé."),
]
labels_attendus = ["Excellent", "Bon", "Faible"]

print("=" * 70)
print("  DÉMO INFÉRENCE — 3 cas test (Expert / Junior / Débutant)")
print("=" * 70)
for (obj_text, resp_text), expected in zip(test_cases, labels_attendus):
    r = predict_full(obj_text, resp_text)
    match = "OK" if r['quality'] == expected else "!"
    print(f"\n  {match}  Attendu: {expected:<12}  Prédit: {r['quality']}")
    print(f"     Overall      : {r['overall_score']:.2f}/10  |  Niveau ALIA : {r['niveau_alia']}")
    print(f"     Scores       : sci={r['scores']['scientific']:.1f} clar={r['scores']['clarity']:.1f} obj={r['scores']['objection']:.1f}")
    print(f"     ACRV         : {r['acrv_score']}/4  |  Conformité : {r['conformite']}  |  Format : {r['visit_format']}")
    print(f"     Sentiment    : {r['sentiment']}  |  Seuil passage : {r['seuil_passage']}/10 → {r['prochain_niveau']}")


  DÉMO INFÉRENCE — 3 cas test (Expert / Junior / Débutant)

  OK  Attendu: Excellent     Prédit: Excellent
     Overall      : 8.82/10  |  Niveau ALIA : Expert
     Scores       : sci=8.8 clar=8.9 obj=8.8
     ACRV         : 4/4  |  Conformité : True  |  Format : Approfondie
     Sentiment    : Positif  |  Seuil passage : 10.0/10 → —

  !  Attendu: Bon           Prédit: Faible
     Overall      : 5.80/10  |  Niveau ALIA : Junior
     Scores       : sci=5.6 clar=5.7 obj=6.0
     ACRV         : 2/4  |  Conformité : True  |  Format : Standard
     Sentiment    : Neutre  |  Seuil passage : 8.0/10 → Confirmé

  OK  Attendu: Faible        Prédit: Faible
     Overall      : 3.55/10  |  Niveau ALIA : Junior
     Scores       : sci=3.8 clar=3.5 obj=3.4
     ACRV         : 0/4  |  Conformité : False  |  Format : Flash
     Sentiment    : Négatif  |  Seuil passage : 8.0/10 → Confirmé


## 9. Conclusions & Architecture retenue

### Résultats benchmarks

| Tâche | Modèle retenu | Métrique clé | Note |
|-------|--------------|--------------|------|
| T1 — Qualité | SVM RBF (C=1.0) | F1-Macro | Classification 3 classes |
| T2 — Scores | Ridge MultiOutput | R² overall | Pondération ALIA 0.25/0.30/0.45 |
| T3 — Sentiment | LogReg (C=1.0) | F1-Macro | 3 classes ambiguës |
| T4 — Niveau ALIA | SVM RBF | F1-Macro | 4 niveaux référentiel officiel |
| T5 — Format visite | LogReg | F1-Macro | Flash/Standard/Approfondie |
| T6 — Conformité | Règle regex + LogReg | F1-Macro | Mots tueurs — règle prioritaire |

### Features les plus discriminantes (top 5)
1. `acrv_completeness` — complétude méthode A-C-R-V
2. `has_mot_tueur` — détection surpromesses
3. `has_bip_signal` — signaux de closing (BIP)
4. `response_quality_proxy` — score composite
5. `has_acrv_v` — étape Validation présente

### Architecture retenue pour `nlp_scoring_train.py V2`
```python
Input    : objection_text + '[SEP]' + rep_response
Features : TF-IDF(8000, ngram 1-2) + 24 features linguistiques → sparse concat
T1       : SVC(kernel='rbf', C=1.0)           → Qualité
T2       : MultiOutputRegressor(Ridge(1.0))   → Scores (sci/clar/obj)
T3       : LogisticRegression(C=1.0)          → Sentiment
T4       : SVC(kernel='rbf', C=1.0)           → Niveau ALIA
T5       : LogisticRegression(C=1.0)          → Format visite
T6       : Règle regex (prioritaire) + LogReg → Conformité
```

### Next steps → `nlp_scoring_train.py V2`
- [OK] Architecture et features validées
- [OK] Baselines établis sur les 6 tâches
- [Soon] GridSearchCV sur T1 et T4
- [Soon] Classe `NLPScoringModel` V2 avec output enrichi (niveau ALIA + conformité)


In [58]:
print("=" * 60)
print("  ARTIFACTS SAUVEGARDÉS — nlp_scoring_exploration V2")
print("=" * 60)
import os
for f in sorted(os.listdir('models')):
    if 'v2' in f:
        sz = os.path.getsize(f'models/{f}') / 1024
        print(f"  {f:<40}  {sz:>7.1f} KB")

print()
for f in sorted(os.listdir('outputs')):
    if f.startswith('nlp_'):
        sz = os.path.getsize(f'outputs/{f}') / 1024
        print(f"  outputs/{f:<38}  {sz:>7.1f} KB")

print()

  ARTIFACTS SAUVEGARDÉS — nlp_scoring_exploration V2
  feature_extractor_v2.pkl                      0.1 KB
  label_encoders_v2.pkl                         1.5 KB
  scaler_feats_v2.pkl                           1.2 KB
  t1_quality_v2.pkl                           191.5 KB
  t2_scores_v2.pkl                             37.3 KB
  t3_sentiment_v2.pkl                          37.1 KB
  t4_niveau_v2.pkl                            424.9 KB
  t5_format_v2.pkl                             37.1 KB
  t6_conformite_v2.pkl                         13.0 KB
  tfidf_v2.pkl                                 56.1 KB

  outputs/nlp_01_distributions.png                  117.4 KB
  outputs/nlp_02_qualite_niveau.png                  77.7 KB
  outputs/nlp_03_acrv_conformite.png                 79.4 KB
  outputs/nlp_04_produit_format.png                 119.7 KB
  outputs/nlp_05_features_by_quality.png            133.5 KB
  outputs/nlp_06_cm_t1.png                           22.7 KB
  outputs/nlp_07_regression_t2